In [1]:
import torch
import os
import sys
from torch import nn
import torchvision as tv
import time
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
from tqdm import tqdm

In [2]:
# device = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
NUM_CLASSES = 50
EPOCHS = 10
LEARNING_RATE = 0.001

device = "cpu"

In [3]:
train_transform = tv.transforms.Compose([
    tv.transforms.Resize((224, 224)),
    tv.transforms.RandomHorizontalFlip(),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = tv.transforms.Compose([
    tv.transforms.Resize((224, 224)),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [4]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        img_id = int(img_name.split('.')[0])
        return image, img_id

In [5]:
train_dir = './hw_4/train_butterflies/train_split/'
test_dir = './hw_4/test_butterflies/valid/'

# train_dataset = ButterflyDataset(train_dir, transform=train_transform)

# test_dataset = ButterflyDataset(test_dir, transform=test_transform, train=False)
test_dataset = TestDataset(test_dir, transform=test_transform)

traint_dataset = tv.datasets.ImageFolder(train_dir, transform=train_transform)
# test_dataset = tv.datasets.ImageFolder(test_dir, transform=test_transform)

# train_iter = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE)

train_size = int(0.7 * len(traint_dataset))
val_size = len(traint_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(traint_dataset, [train_size, val_size])

train_iter = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_iter = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_iter = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [ ]:
model1 = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),  # 224 -> 112
        nn.ReLU(inplace=True),
        
        nn.Conv2d(16, 64, kernel_size=3, stride=2, padding=1),  # 112 -> 56
        nn.ReLU(inplace=True),
        
        nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 56 -> 28
        nn.ReLU(inplace=True),
        
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(128, 64),
        nn.ReLU(inplace=True),
        nn.Linear(64, 50)
)

In [7]:
model1 = model1.to(device)

In [8]:
def evaluate_accuracy(data_iter, net):
    acc_sum, n = torch.Tensor([0]), 0
    for X, y in data_iter:
        acc_sum += (net(X).argmax(axis=1) == y).sum()
        n += y.shape[0]
    return acc_sum.item() / n

In [9]:
def train(net, train_iter, test_iter, optimizer, num_epochs):
    loss = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        train_l_sum, train_acc_sum, n, start = 0.0, 0.0, 0, time.time()
        
        progress_bar = tqdm(train_iter, desc=f'Epoch {epoch+1}/{num_epochs}')

        for X, y in progress_bar:
            optimizer.zero_grad() # обнуляем градиенты
            y_hat = net(X) # предсказываем значения
            l = loss(y_hat, y) # счтаем функцию потерь
            l.backward() # считаем градиенты, которые останутся "на парметрах"
            optimizer.step() # делаем шаг, градиент в параметрах
            train_l_sum += l.item()
            train_acc_sum += (y_hat.argmax(axis=1) == y).sum().item()
            n += y.shape[0]
            
            progress_bar.set_postfix({
                'loss': f'{l.item():.4f}',
                'acc': f'{(train_acc_sum / n):.4f}'
            })

        test_acc = evaluate_accuracy(test_iter, net)
        print(f'epoch {epoch + 1}, loss {train_l_sum / n:.4f}, train acc {train_acc_sum / n:.3f}' \
              f', test acc {test_acc:.3f}, time {time.time() - start:.1f} sec')

In [10]:
optimizer = torch.optim.Adam(model1.parameters(), lr=LEARNING_RATE)

In [11]:
train(model1, train_iter, val_iter, optimizer, EPOCHS)

Epoch 1/10: 100%|██████████| 109/109 [01:33<00:00,  1.17it/s, loss=3.6649, acc=0.0392]


epoch 1, loss 0.1193, train acc 0.039, test acc 0.051, time 109.2 sec


Epoch 2/10: 100%|██████████| 109/109 [01:22<00:00,  1.32it/s, loss=3.1920, acc=0.0888]


epoch 2, loss 0.1098, train acc 0.089, test acc 0.114, time 95.2 sec


Epoch 3/10: 100%|██████████| 109/109 [01:14<00:00,  1.47it/s, loss=2.8272, acc=0.1393]


...

epoch 8, loss 0.0837, train acc 0.259, test acc 0.251, time 94.5 sec


Epoch 9/10: 100%|██████████| 109/109 [01:39<00:00,  1.09it/s, loss=2.7831, acc=0.2719]


epoch 9, loss 0.0814, train acc 0.272, test acc 0.278, time 120.8 sec


Epoch 10/10: 100%|██████████| 109/109 [01:32<00:00,  1.18it/s, loss=2.2665, acc=0.2970]


epoch 10, loss 0.0785, train acc 0.297, test acc 0.297, time 106.9 sec


In [12]:
model1.eval()
predictions = []
pred_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_iter, desc="Predicting"):
        images = images.to(device)
        outputs = model1(images)
        _, predicted = torch.max(outputs, 1)
        
        predictions.extend(predicted.cpu().numpy())
        pred_ids.extend(ids.numpy())

# Save submission file
real_classes = []
for pred_idx in predictions:
    folder_name = traint_dataset.classes[pred_idx] 
    class_num = int(folder_name.replace('class_', '')) 
    real_classes.append(class_num)

submission = pd.DataFrame({'index': pred_ids, 'label': real_classes})
submission = submission.sort_values('index')
submission.to_csv('submission.csv', index=False)

Predicting: 100%|██████████| 8/8 [00:02<00:00,  2.96it/s]


In [13]:
model2 = tv.models.resnet18(pretrained=True)

num_ftrs = model2.fc.in_features
model2.fc = nn.Linear(num_ftrs, 50)

In [14]:
optimizer = torch.optim.Adam(model2.parameters(), lr=LEARNING_RATE)

In [16]:
train(model2, train_iter, val_iter, optimizer, 4)

Epoch 1/4: 100%|██████████| 109/109 [08:47<00:00,  4.84s/it, loss=1.1756, acc=0.8270]


epoch 1, loss 0.0192, train acc 0.827, test acc 0.812, time 602.0 sec


Epoch 2/4: 100%|██████████| 109/109 [08:20<00:00,  4.60s/it, loss=0.6693, acc=0.8697]


epoch 2, loss 0.0138, train acc 0.870, test acc 0.839, time 574.9 sec


Epoch 3/4: 100%|██████████| 109/109 [08:14<00:00,  4.53s/it, loss=0.1428, acc=0.9083]


epoch 3, loss 0.0101, train acc 0.908, test acc 0.827, time 575.3 sec


Epoch 4/4: 100%|██████████| 109/109 [08:11<00:00,  4.51s/it, loss=0.7076, acc=0.9219]


epoch 4, loss 0.0076, train acc 0.922, test acc 0.874, time 560.0 sec


In [20]:
test_dataset = TestDataset(test_dir, transform=test_transform)
test_iter = DataLoader(test_dataset, batch_size=BATCH_SIZE)


In [26]:
model2.eval()
predictions = []
pred_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_iter, desc="Predicting"):
        images = images.to(device)
        outputs = model2(images)
        _, predicted = torch.max(outputs, 1)
        
        predictions.extend(predicted.numpy())
        pred_ids.extend(ids.numpy())

print(predictions)

# Save submission file
real_classes = []
for pred_idx in predictions:
    folder_name = traint_dataset.classes[pred_idx] 
    class_num = int(folder_name.replace('class_', '')) 
    real_classes.append(class_num)

submission = pd.DataFrame({'index': pred_ids, 'label': real_classes})
submission.to_csv('submission.csv', index=False)

Predicting: 100%|██████████| 8/8 [00:20<00:00,  2.59s/it]

[np.int64(4), np.int64(45), np.int64(12), np.int64(0), np.int64(22), np.int64(28), np.int64(27), np.int64(45), np.int64(24), np.int64(41), np.int64(14), np.int64(48), np.int64(42), np.int64(18), np.int64(41), np.int64(8), np.int64(40), np.int64(16), np.int64(45), np.int64(29), np.int64(23), np.int64(46), np.int64(33), np.int64(11), np.int64(21), np.int64(26), np.int64(9), np.int64(47), np.int64(10), np.int64(19), np.int64(15), np.int64(31), np.int64(46), np.int64(47), np.int64(5), np.int64(17), np.int64(4), np.int64(25), np.int64(42), np.int64(6), np.int64(7), np.int64(15), np.int64(8), np.int64(33), np.int64(22), np.int64(30), np.int64(38), np.int64(34), np.int64(32), np.int64(10), np.int64(15), np.int64(28), np.int64(43), np.int64(44), np.int64(19), np.int64(31), np.int64(46), np.int64(11), np.int64(23), np.int64(31), np.int64(27), np.int64(20), np.int64(10), np.int64(38), np.int64(42), np.int64(40), np.int64(2), np.int64(41), np.int64(8), np.int64(5), np.int64(34), np.int64(45), np.